# 03 · La misma web React — opcional, no un tercer paso obligatorio

Si ya usas la web en tu PC, **no necesitas este notebook**: 01 prepara datos y 02 entrega workspace+modelo.
Ésta es una forma alternativa de iniciar la aplicación **exacta del mismo release** en un runtime Colab/Jupyter.
No se crea otra UI, no se importa Gradio ni se duplican sus componentes. El servidor es `webapp_react/server.py` y el
frontend es `webapp_react/frontend/dist` de esa entrega, con todas sus limitaciones y controles de evidencia originales.

La vía recomendada sigue siendo el PC local: conserva mejor los artefactos, permite asociar los MP4 originales y no depende
de la vida de una VM Colab. El visor de puertos de Colab es una facilidad opcional de ese runtime, no una publicación
permanente ni un túnel externo. Aquí sólo se verificaron sus llamadas con fixtures, no acceso real a una GPU/T4/Drive.


## Abrir el mismo release privado: ZIP subido, carpeta ya extraída o clon fijado por commit

Necesitas dos cosas diferentes: **el ZIP de la entrega FENCINGLY** (código de confianza) y, a partir del segundo notebook,
**el ZIP de workspace** (sólo datos/modelo). No son intercambiables. No subas un workspace en el lugar del release.
El código de esta entrega no tiene por qué existir en ningún repositorio remoto. No hay una descarga de GitHub por defecto.

La entrega recomendada tiene `FENCINGLY/webapp/fencingly_v2/` y `FENCINGLY/webapp/webapp_react/frontend/dist/`.
Los notebooks viven en `FENCINGLY/colab/notebooks/` y el estudio en `FENCINGLY/study/`. Después del bootstrap, **ROOT apunta
al contenido de `webapp/`**, no a su carpeta padre ni al workspace de datos.

**En Colab:** abre este `.ipynb` con Archivo → Subir notebook. Si quieres acelerar la extracción/entrenamiento, selecciona
Entorno de ejecución → Cambiar tipo de entorno → GPU. Colab decide qué GPU hay; una T4 no está garantizada. Sube el ZIP del
release cuando se solicite, o escribe su ruta en `RELEASE_ZIP` si está en Drive ya montado. Para un fichero grande, monta
Drive primero en una celda propia con `google.colab.drive.mount('/content/drive')` y usa su ruta, evitando cargarlo entero
con el navegador. Si conservas el código extraído en este mismo runtime, proporciona `RELEASE_ROOT` y no vuelvas a extraer.

**En un PC/Jupyter:** descomprime la entrega de confianza y fija `RELEASE_ROOT` a su carpeta `webapp` (por ejemplo
`C:/mis_datos/FENCINGLY/webapp`). No uses una ruta de esta auditoría en otro ordenador. Todos los paths posteriores se
calculan desde ese ROOT y desde tus carpetas explícitas.

La celda mínima de bootstrap sólo lee del ZIP un helper **cuyo SHA-256 está fijado en este notebook**. No ejecuta un helper
arbitrario procedente de un workspace. Ese helper valida las rutas de todo el release, tamaños, expansión, colisiones,
enlaces y presencia del servidor/React `dist` antes de extraer a una carpeta nueva. No usa `extractall`. El estudio y los
notebooks del ZIP completo no se duplican dentro de `webapp/`. Si tienes el SHA-256 publicado **junto a tu entrega privada**,
fíjalo en `EXPECTED_RELEASE_SHA256`: integridad sin una referencia de confianza no equivale a autenticidad del autor.
No cambies el hash del helper para hacer pasar un ZIP desconocido. Si actualizas el código, pide el notebook del mismo release.

### Tercera vía opcional: clonar un repositorio con el commit fijado

Si prefieres no volver a subir el ZIP en cada sesión, fija `FENCINGLY_RELEASE_GIT` a la URL HTTPS del repositorio,
`FENCINGLY_RELEASE_GIT_REF` a su rama o etiqueta y `FENCINGLY_RELEASE_COMMIT` al commit completo de 40 hex que hayas
revisado. Por defecto esas tres variables están vacías y **no se clona nada**: la vía por defecto sigue siendo el ZIP
privado subido a mano, y el flujo sin red externa no cambia. Publicar el código sigue siendo una decisión tuya, no un
requisito de estos notebooks.

El clon es superficial (`--depth 1`) sobre una carpeta nueva y **no relaja la integridad**. Una rama es móvil, así que
el pin de commit es **obligatorio** al clonar: si el `HEAD` clonado no es exactamente ese commit, la celda se detiene.
Después se comprueba además el SHA-256 del helper `fencingly_v2/artifact_bundle.py` contra `HELPER_SHA256`, igual que
en la vía del ZIP. Un commit fijado prueba qué árbol se ejecuta, no quién lo escribió: revisa el repositorio antes de
fijarlo y no actualices el pin sin leer el diff. No se hace `push`, no se publican datos y no se sube nada al clonar.

El repositorio debe exponer el `webapp` en **una sola** de estas tres formas, las mismas que acepta el ZIP:
`FENCINGLY/webapp/`, `webapp/` o la propia raíz. Dentro tienen que estar `fencingly_v2/`, `webapp_react/server.py` y
`webapp_react/frontend/dist/index.html` ya compilado; sin ese `dist` la celda se detiene y no se arranca otro frontend.

La procedencia elegida queda en `RELEASE_PROVENANCE` (origen, URL, ref, commit, hash del helper y si el pin coincidió),
se imprime y viaja con el workspace exportado.


In [ ]:
import os, sys, hashlib, shutil, subprocess, zipfile, types
from pathlib import Path
from datetime import datetime, timezone

RELEASE_ROOT = os.environ.get("FENCINGLY_RELEASE_ROOT", "")
RELEASE_ZIP = os.environ.get("FENCINGLY_RELEASE_ZIP", "")
RELEASE_GIT_URL = os.environ.get("FENCINGLY_RELEASE_GIT", "")
RELEASE_GIT_REF = os.environ.get("FENCINGLY_RELEASE_GIT_REF", "main")
EXPECTED_RELEASE_COMMIT = os.environ.get("FENCINGLY_RELEASE_COMMIT", "")
EXPECTED_RELEASE_SHA256 = os.environ.get("FENCINGLY_RELEASE_SHA256", "")
IN_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules)
HELPER_SHA256 = "ee0e2a0c5cebf80c41c9b5e15c43c48801173780682553e7f24bd9bbb58848e3"
RELEASE_LAYOUTS = ("FENCINGLY/webapp", "webapp", "")

if RELEASE_ROOT:
    ROOT = Path(RELEASE_ROOT).expanduser().resolve()
    RELEASE_PROVENANCE = {"source": "release_root", "location": str(ROOT), "helper_sha256": HELPER_SHA256,
                          "authenticity": "user_must_trust_release_source"}
elif RELEASE_GIT_URL:
    if not RELEASE_GIT_URL.startswith("https://") or any(character.isspace() for character in RELEASE_GIT_URL):
        raise ValueError("Sólo se clona un remoto HTTPS explícito; no se aceptan otros transportes")
    if not RELEASE_GIT_REF or RELEASE_GIT_REF.startswith("-") or any(character.isspace() for character in RELEASE_GIT_REF):
        raise ValueError("Indica una rama o etiqueta explícita en FENCINGLY_RELEASE_GIT_REF")
    if len(EXPECTED_RELEASE_COMMIT) != 40 or any(character not in "0123456789abcdef" for character in EXPECTED_RELEASE_COMMIT.lower()):
        raise ValueError("Clonar exige FENCINGLY_RELEASE_COMMIT con el commit completo de 40 hex que hayas revisado: una rama es móvil y no acredita integridad")
    if shutil.which("git") is None:
        raise ValueError("Este runtime no trae git; usa RELEASE_ZIP o RELEASE_ROOT")
    parent = Path("/content" if IN_COLAB else Path.cwd())
    clone_root = parent / ("fencingly_clone_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    if clone_root.exists():
        raise ValueError("El destino del clon ya existe; no se reutiliza ni se sobrescribe una carpeta previa")
    cloned = subprocess.run(["git", "clone", "--depth", "1", "--branch", RELEASE_GIT_REF, "--", RELEASE_GIT_URL, str(clone_root)],
                            capture_output=True, text=True)
    if cloned.returncode != 0 or not (clone_root / ".git").is_dir():
        raise ValueError("El clon superficial ha fallado: " + " ".join((cloned.stderr or "").split())[:300])
    resolved = subprocess.run(["git", "-C", str(clone_root), "rev-parse", "HEAD"], capture_output=True, text=True)
    cloned_commit = (resolved.stdout or "").strip().lower()
    if resolved.returncode != 0 or len(cloned_commit) != 40:
        raise ValueError("No se ha podido leer el commit clonado; no se ejecuta un árbol sin identificar")
    if cloned_commit != EXPECTED_RELEASE_COMMIT.lower():
        raise ValueError("El commit clonado no coincide con FENCINGLY_RELEASE_COMMIT: revisa el cambio y actualiza el pin; no se ejecuta un árbol no fijado")
    layouts = [clone_root.joinpath(*[part for part in layout.split("/") if part]) for layout in RELEASE_LAYOUTS]
    layouts = [candidate for candidate in layouts if (candidate / "fencingly_v2" / "artifact_bundle.py").is_file()]
    if len(layouts) != 1:
        raise ValueError("El repositorio debe exponer exactamente un webapp con fencingly_v2/artifact_bundle.py en FENCINGLY/webapp, en webapp o en la raíz")
    ROOT = layouts[0].resolve()
    cloned_helper = (ROOT / "fencingly_v2" / "artifact_bundle.py").read_bytes()
    if hashlib.sha256(cloned_helper).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper clonado no coincide con este notebook: usa el commit del mismo release; no omitas el control")
    RELEASE_PROVENANCE = {"source": "git_clone", "url": RELEASE_GIT_URL, "ref": RELEASE_GIT_REF, "commit": cloned_commit,
                          "helper_sha256": HELPER_SHA256, "authenticity": "pinned_commit_matched"}
    print("Release clonado y fijado:", RELEASE_PROVENANCE)
else:
    if not RELEASE_ZIP:
        if not IN_COLAB:
            raise ValueError("Fija RELEASE_ROOT al webapp del release, RELEASE_ZIP al ZIP privado o RELEASE_GIT al repositorio con su commit fijado")
        from google.colab import files
        uploaded_release = files.upload()
        candidates = [name for name in uploaded_release if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise ValueError("Sube exactamente un ZIP de RELEASE, no el workspace")
        RELEASE_ZIP = candidates[0]
        del uploaded_release
    if Path(RELEASE_ZIP).stat().st_size > 4 * 1024 ** 3:
        raise ValueError("Release mayor que el presupuesto de 4 GiB")
    with Path(RELEASE_ZIP).open("rb") as release_stream:
        footer = zipfile._EndRecData(release_stream)
        if footer is None or footer[zipfile._ECD_ENTRIES_TOTAL] > 30000 or footer[zipfile._ECD_SIZE] > 8 * 1024 ** 2:
            raise ValueError("Metadatos del ZIP de release excesivos o inválidos antes de cargar su directorio")
    with zipfile.ZipFile(RELEASE_ZIP) as archive:
        if len(archive.infolist()) > 30000:
            raise ValueError("Release con demasiadas entradas")
        approved_names = {prefix + "fencingly_v2/artifact_bundle.py" for prefix in ("FENCINGLY/webapp/", "webapp/", "")}
        helpers = [entry for entry in archive.infolist() if entry.filename in approved_names]
        if len(helpers) != 1 or helpers[0].file_size > 150000 or helpers[0].flag_bits & 1:
            raise ValueError("Helper del release ausente, ambiguo o excesivo")
        helper_bytes = archive.read(helpers[0])
    if hashlib.sha256(helper_bytes).hexdigest() != HELPER_SHA256:
        raise ValueError("El helper no coincide con este notebook: usa el release correspondiente; no omitas el control")
    bootstrap = types.ModuleType("_fencingly_verified_release_bootstrap")
    sys.modules[bootstrap.__name__] = bootstrap
    exec(compile(helper_bytes, "verified_artifact_bundle.py", "exec"), bootstrap.__dict__)
    parent = Path("/content" if IN_COLAB else Path.cwd())
    destination = parent / ("fencingly_release_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f"))
    extracted = bootstrap.extract_release_to_new_directory(RELEASE_ZIP, destination,
                    expected_sha256=EXPECTED_RELEASE_SHA256 or None)
    ROOT = Path(extracted["root"])
    RELEASE_PROVENANCE = {"source": "release_zip", "sha256": extracted["sha256"], "webapp_prefix": extracted["webapp_prefix"],
                          "helper_sha256": HELPER_SHA256, "authenticity": extracted["authenticity"]}
    print("Release validado:", extracted)

required = ("fencingly_v2/colab_workflow.py", "fencingly_v2/training_recipe.py", "fencingly_v2/epoch_checkpoint.py",
            "fencingly_v2/measurement_artifacts.py", "webapp_react/server.py", "webapp_react/frontend/dist/index.html")
if not all((ROOT / relative).is_file() for relative in required):
    raise ValueError("ROOT no contiene el webapp completo del mismo release, incluido React dist")
if "fencingly_v2" in sys.modules and Path(sys.modules["fencingly_v2"].__file__).resolve().parent != ROOT / "fencingly_v2":
    raise RuntimeError("Otro release ya está importado: reinicia el kernel; no mezcles bindings de fuentes distintas")
sys.path.insert(0, str(ROOT))
os.environ["FENCINGLY_RELEASE_ROOT"] = str(ROOT)
from fencingly_v2 import colab_workflow as CW, artifact_bundle as AB
print("ROOT del código compartido:", ROOT)


## Dependencias: comprobar antes de instalar

La entrega no presupone que Colab traiga las versiones correctas. El informe muestra Python y módulos realmente
instalados. La instalación está **desactivada**. Si falta algo, activa `INSTALL_MISSING=True` de forma consciente;
el helper sólo solicita los módulos de esa fase. Nunca instala automáticamente torch/CUDA cuando torch falta: elige
el runtime y la distribución oficial adecuados, reinicia si el instalador lo pide y repite la comprobación. No copies un
entorno virtual de Windows a Colab. No se ha instalado aquí un stack GPU para aparentar una prueba en CUDA.

Un import correcto no verifica la compatibilidad de un Store o modelo: esos artefactos conservan sus propios contratos.
La parte web puede necesitar dependencias opcionales adicionales para informes/PDF; usa las requirements de la misma
entrega y los mensajes del servidor, no una lista de paquetes de otro notebook histórico.


In [ ]:
INSTALL_MISSING = False
print(CW.dependency_report("web"))
if INSTALL_MISSING:
    CW.install_missing("web", approved=True)
RUN_WEB = os.environ.get("FENCINGLY_RUN_WEB", "0") == "1"
WORKSPACE_ZIP = os.environ.get("FENCINGLY_WORKSPACE_ZIP", "")
EXISTING_WORKSPACE = os.environ.get("FENCINGLY_WORKSPACE", "")
LOCAL_CLIPS_DIRS = []
WEAPON = os.environ.get("FENCINGLY_WEAPON", "foil")
PORT = 8000
FIE_ONLINE = False
SERVER = None
WORK = Path(EXISTING_WORKSPACE) if EXISTING_WORKSPACE else None


## Restaurar datos sin ejecutar su contenido ni sobreescribir nada

Usa el ZIP final de 02 o un workspace de evidencia de 01. Si está en Drive, móntalo conscientemente antes de proporcionar
su ruta; el notebook no copia vídeos originales ni busca por todo tu Drive. En el mismo runtime puedes indicar
`EXISTING_WORKSPACE` para reutilizar tu carpeta anterior. No descomprime un ZIP encima de ella.

Se validan allowlist, hashes, nombres portables, ausencia de enlaces/traversal, límites de expansión y tamaños antes de
crear el nuevo destino. La restauración **no importa Python del ZIP, no unpicklea pesos, no instala dependencias y no adopta
el snapshot de splits como autoridad global**. Después se carga el modelo exclusivamente mediante el loader del release.
El formato portable actual verifica CPU FP32; si no lo acepta, el error se muestra y no se fuerza un fallback legacy.

`--no-fie` y `FENCINGLY_ALLOW_REMOTE_VIDEO=0` mantienen la red externa desactivada por defecto. Para ver vídeo original,
apunta `LOCAL_CLIPS_DIRS` a carpetas autorizadas. No basta con que un registro del JSON tenga una URL para afirmar que su
vídeo está disponible. La ausencia de original no convierte un replay de keypoints en vídeo real ni en simulación 3D.

`WEAPON` sigue el mismo `FENCINGLY_WEAPON` que el notebook 01 y por defecto es `foil`. El workspace guarda su manifiesto como `manifest_<arma>.csv`, así que un workspace de espada o sable exige fijar esa misma variable aquí; si no coincide, el helper se detiene con «Manifest is missing» y no se inventa otro manifiesto ni se sirve un arma distinta de la que se procesó.


In [ ]:
if RUN_WEB:
    parent = Path("/content/fencingly_runs" if IN_COLAB else ROOT.parent / "colab_runs")
    parent.mkdir(parents=True, exist_ok=True)
    if WORKSPACE_ZIP:
        restored = AB.restore_to_new_directory(WORKSPACE_ZIP, parent / ("web_" + CW.stamp()))
        WORK = Path(restored["destination"])
    if WORK is None or not WORK.is_dir():
        raise ValueError("Configura WORKSPACE_ZIP o EXISTING_WORKSPACE; no se crea una demo vacía como si tuviera datos")
    model_path = WORK / "model_bundle.zip"
    if model_path.is_file():
        from fencingly_v2 import train as TR, feature_contract as FC
        checked_model, checked_meta = TR.load_bundle(model_path, device="cpu")
        print("Contrato del modelo:", FC.resolve_contract(checked_model)["training_intent"])
        print("Verificación portable local:", getattr(checked_model, "portable_verification", None))
        del checked_model
    else:
        model_path = None
        print("Workspace de evidencia sin modelo: no se inventan predicciones")
    COMMAND = CW.shared_web_command(ROOT, WORK, port=PORT, bundle=model_path,
        manifest=WORK / f"manifest_{WEAPON}.csv", clips_dirs=LOCAL_CLIPS_DIRS, fie_online=FIE_ONLINE)
    import shlex, subprocess
    print("Comando exacto del mismo frontend:")
    print(subprocess.list2cmdline(COMMAND) if sys.platform == "win32" else shlex.join(COMMAND))
else:
    print("Modo seguro: ningún servidor ni puerto se han abierto")


## Iniciar el servidor y abrir el puerto del kernel

Se escucha en `127.0.0.1`, no en una dirección pública. El helper conserva un log nuevo bajo `WORK/checkpoints/`, espera
HTTP 200 en `/`, comprueba que el proceso siga vivo y exige que el HTML coincida byte a byte con el `index.html` del release.
Un puerto ya ocupado se rechaza: no se reutiliza ni se detiene un servidor ajeno. No arranca un build React diferente ni requiere Node cuando el
`dist` ya viene compilado. Si falta `dist/index.html`, detiene la ejecución: un Gradio no es un reemplazo equivalente.

En Colab se intenta `google.colab.output.serve_kernel_port_as_window` y, si esa API no existe, su variante iframe. Si el
runtime no ofrece ninguna, usa la aplicación local. Es una comprobación de disponibilidad, no una promesa de que todos
los navegadores/proxies de Colab tengan el mismo comportamiento. No se instala ngrok ni se pide publicar GitHub.

En PC abre la URL local que imprime. Si el servidor devuelve un error, lee el log antes de culpar al navegador. Una
pantalla sin errores JS no valida la precisión de pose/luces, y un receipt de compatibilidad no valida etiquetas deportivas.


In [ ]:
if RUN_WEB:
    SERVER = CW.start_shared_web(ROOT, WORK, port=PORT, bundle=model_path,
        manifest=WORK / f"manifest_{WEAPON}.csv", clips_dirs=LOCAL_CLIPS_DIRS, fie_online=FIE_ONLINE)
    print("URL:", SERVER["url"], "| log:", SERVER["log"])
    if IN_COLAB:
        CW.show_colab_web(PORT)
    else:
        print("Abre esa URL en el navegador de tu PC; no hay otro frontend")


## Comprobación manual que sí debes hacer con tus artefactos

1. Busca un tirador y comprueba cuántos clips están registrados frente a procesados/evaluables. No confundas 140.739
registros del JSON con descargas locales ni con muestras elegibles del modelo.
2. Abre un clip y verifica arma, fuente, fecha, unidades, cobertura y advertencias. Un null no es cero. Los segmentos/gaps
no deben dibujarse como trayectorias continuas ni interpolarse para fabricar una acción.
3. Si hay MP4 local autorizado, reprodúcelo a petición; sin MP4, el sistema debe explicitar qué evidencia procede sólo de NPZ.
4. Distingue historial FIE publicado de grupos locales parciales. Un tanteo observado 14–14 no identifica por sí solo el
final/ganador de un asalto ni un relevo. Las categorías/formatos y los desconocidos permanecen separados.
5. Si usas el modelo, comprueba el receipt/contrato y las abstenciones; ninguna secuencia totalmente inválida debe obtener
un acierto fabricado. El endpoint sigue siendo post-tocado, no forecasting.
6. Conserva logs de fallos y el ZIP original. No cambies metadata para ocultar un gate de compatibilidad.

Los tests automatizados entregados prueban contratos/fixtures en CPU. La evidencia de navegador e integración del servidor
corresponde al release final y a sus propios informes. No se declara aquí una inspección visual real de Colab que no se haya
hecho, ni ganancias de accuracy o latencia sin una medición separada.


In [ ]:
STOP_SERVER = False
if STOP_SERVER and SERVER is not None:
    SERVER["process"].terminate()
    SERVER["process"].wait(timeout=10)
    print("Servidor de esta llamada detenido; workspace, ZIPs y originales intactos")


## Cierre y privacidad

Antes de cerrar Colab confirma que el ZIP final está en Drive o en tu PC. `STOP_SERVER=True` sólo termina el proceso que
este notebook inició, no borra datos ni cambia servicios de la máquina. No hay publicación de Git, push, captura en una KB
global, cambios de seguridad ni subagentes dentro del flujo. Comparte únicamente los artefactos que decidas compartir y
respeta permisos de vídeos y datos. La aplicación local usa el mismo código privado de esta entrega.
